# Unsupervised Learning: Topic Modeling and Word Embeddings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from bertopic import BERTopic
import nltk
from nltk.tokenize import word_tokenize

# Custom modules
import sys
sys.path.append('../')
from utils.preprocessing import preprocess_text

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# Load processed data
df = pd.read_csv('../data/processed/cleaned_reviews.csv')
print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
# Topic Modeling with BERTopic
docs = df['cleaned_review'].tolist()

topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(docs)

# Get topic info
topic_info = topic_model.get_topic_info()
print("Topic Information:")
display(topic_info.head(10))

# Visualize topics
topic_model.visualize_topics()
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
# Word Embeddings: Train Word2Vec
sentences = [doc.split() for doc in df['cleaned_review']]

# Train Word2Vec model
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)
print(f"Vocabulary size: {len(w2v_model.wv)}")

# Save model
w2v_model.save('../models/word2vec.model')
print("Word2Vec model saved.")

In [ ]:
# Visualize Word Embeddings
words = list(w2v_model.wv.index_to_key)[:100]  # Top 100 words
vectors = np.array([w2v_model.wv[word] for word in words])

# PCA
pca = PCA(n_components=2)
vectors_pca = pca.fit_transform(vectors)

plt.figure(figsize=(12, 8))
plt.scatter(vectors_pca[:, 0], vectors_pca[:, 1])
for i, word in enumerate(words):
    plt.annotate(word, (vectors_pca[i, 0], vectors_pca[i, 1]))
plt.title('Word Embeddings Visualization (PCA)')
plt.show()

# t-SNE
tsne = TSNE(n_components=2, random_state=42)
vectors_tsne = tsne.fit_transform(vectors)

plt.figure(figsize=(12, 8))
plt.scatter(vectors_tsne[:, 0], vectors_tsne[:, 1])
for i, word in enumerate(words):
    plt.annotate(word, (vectors_tsne[i, 0], vectors_tsne[i, 1]))
plt.title('Word Embeddings Visualization (t-SNE)')
plt.show()

In [ ]:
# Distance Metrics
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean

# Example words
word1 = 'good'
word2 = 'great'
word3 = 'bad'

if word1 in w2v_model.wv and word2 in w2v_model.wv and word3 in w2v_model.wv:
    vec1 = w2v_model.wv[word1]
    vec2 = w2v_model.wv[word2]
    vec3 = w2v_model.wv[word3]

    # Cosine similarity
    cos_sim_12 = cosine_similarity([vec1], [vec2])[0][0]
    cos_sim_13 = cosine_similarity([vec1], [vec3])[0][0]

    print(f"Cosine similarity between '{word1}' and '{word2}': {cos_sim_12:.4f}")
    print(f"Cosine similarity between '{word1}' and '{word3}': {cos_sim_13:.4f}")

    # Euclidean distance
    euc_dist_12 = euclidean(vec1, vec2)
    euc_dist_13 = euclidean(vec1, vec3)

    print(f"Euclidean distance between '{word1}' and '{word2}': {euc_dist_12:.4f}")
    print(f"Euclidean distance between '{word1}' and '{word3}': {euc_dist_13:.4f}")

# Semantic search example
def find_similar_reviews(query, df, model, top_n=5):
    query_vec = np.mean([model.wv[word] for word in query.split() if word in model.wv], axis=0)
    similarities = []
    for idx, review in enumerate(df['cleaned_review']):
        review_vec = np.mean([model.wv[word] for word in review.split() if word in model.wv], axis=0)
        if review_vec.shape == query_vec.shape:
            sim = cosine_similarity([query_vec], [review_vec])[0][0]
            similarities.append((idx, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

# Example semantic search
query = "fast claims"
similar = find_similar_reviews(query, df, w2v_model)
print(f"\nTop 5 reviews similar to '{query}':")
for idx, sim in similar:
    print(f"Similarity: {sim:.4f} - {df.iloc[idx]['review']}")